# Imports

In [1]:
import xarray as xr
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client

# PBS

In [2]:
cluster = PBSCluster(
    cores=4, # The number of cores you want
    memory='96GB', # Amount of memory (resource_spec is the one)
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    local_directory='$TMPDIR', # Use your local directory
    account='P93300313', # Input your project ID here
    walltime='02:00:00', # Amount of wall time
)
cluster.scale(jobs=5)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.189:40423,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [3]:
# chunks='auto' appproaches this value
dask.config.set({'array.chunk-size': '8 GiB'})

# Load

In [4]:
Uds = xr.open_zarr('/glade/work/acruz/E3SMv2LE/E3SMv2le_tropic_U', decode_times=True)
Uds = Uds.chunk(chunks='auto')
Uds

<xarray.Dataset> Size: 42GB
Dimensions:  (member: 21, time: 1980, lev: 72, lat: 22, lon: 161)
Coordinates:
  * time     (time) object 16kB 1850-02-01 00:00:00 ... 2015-01-01 00:00:00
  * lev      (lev) float64 576B 0.1238 0.1828 0.2699 ... 986.2 993.8 998.5
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Dimensions without coordinates: member
Data variables:
    OMEGA    (member, time, lev, lat, lon) float32 42GB dask.array<chunksize=(21, 400, 72, 22, 161), meta=np.ndarray>

In [5]:
Wds = xr.open_zarr('/glade/work/acruz/E3SMv2LE/E3SMv2le_tropic_OMEGA')
Wds = Wds.chunk(chunks='auto')
Wds

<xarray.Dataset> Size: 42GB
Dimensions:  (member: 21, time: 1980, lev: 72, lat: 22, lon: 161)
Coordinates:
  * time     (time) object 16kB 1850-02-01 00:00:00 ... 2015-01-01 00:00:00
  * lev      (lev) float64 576B 0.1238 0.1828 0.2699 ... 986.2 993.8 998.5
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Dimensions without coordinates: member
Data variables:
    U        (member, time, lev, lat, lon) float32 42GB dask.array<chunksize=(21, 400, 72, 22, 161), meta=np.ndarray>

# Calculate

In [6]:
Uclim = Uds.groupby('time.month').mean(dim='time')
Uclim = Uclim.chunk(chunks='auto')
Uclim

<xarray.Dataset> Size: 257MB
Dimensions:  (month: 12, member: 21, lev: 72, lat: 22, lon: 161)
Coordinates:
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * lev      (lev) float64 576B 0.1238 0.1828 0.2699 ... 986.2 993.8 998.5
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Dimensions without coordinates: member
Data variables:
    OMEGA    (month, member, lev, lat, lon) float32 257MB dask.array<chunksize=(12, 21, 72, 22, 161), meta=np.ndarray>

In [7]:
Uanom = Uds.groupby('time.month') - Uclim
# faster when saving 10+GiB to use zarr and rechunk
Uanom = Uanom.chunk(chunks='auto')
Uanom

<xarray.Dataset> Size: 42GB
Dimensions:  (lat: 22, lev: 72, lon: 161, time: 1980, member: 21)
Coordinates:
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lev      (lev) float64 576B 0.1238 0.1828 0.2699 ... 986.2 993.8 998.5
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
  * time     (time) object 16kB 1850-02-01 00:00:00 ... 2015-01-01 00:00:00
    month    (time) int64 16kB dask.array<chunksize=(1980,), meta=np.ndarray>
Dimensions without coordinates: member
Data variables:
    OMEGA    (member, time, lev, lat, lon) float32 42GB dask.array<chunksize=(21, 400, 72, 22, 161), meta=np.ndarray>

In [8]:
Uanom.to_zarr('/glade/work/acruz/E3SMv2LE/anomalies/E3SM_tropic_U', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [9]:
Wclim = Wds.groupby('time.month').mean(dim='time')
Wclim = Wclim.chunk(chunks='auto')
Wclim

<xarray.Dataset> Size: 257MB
Dimensions:  (month: 12, member: 21, lev: 72, lat: 22, lon: 161)
Coordinates:
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * lev      (lev) float64 576B 0.1238 0.1828 0.2699 ... 986.2 993.8 998.5
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Dimensions without coordinates: member
Data variables:
    U        (month, member, lev, lat, lon) float32 257MB dask.array<chunksize=(12, 21, 72, 22, 161), meta=np.ndarray>

In [10]:
Wanom = Wds.groupby('time.month') - Wclim
Wanom = Wanom.chunk(chunks='auto')
Wanom

<xarray.Dataset> Size: 42GB
Dimensions:  (lat: 22, lev: 72, lon: 161, time: 1980, member: 21)
Coordinates:
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lev      (lev) float64 576B 0.1238 0.1828 0.2699 ... 986.2 993.8 998.5
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
  * time     (time) object 16kB 1850-02-01 00:00:00 ... 2015-01-01 00:00:00
    month    (time) int64 16kB dask.array<chunksize=(1980,), meta=np.ndarray>
Dimensions without coordinates: member
Data variables:
    U        (member, time, lev, lat, lon) float32 42GB dask.array<chunksize=(21, 400, 72, 22, 161), meta=np.ndarray>

In [11]:
Wanom.to_zarr('/glade/work/acruz/E3SMv2LE/anomalies/E3SM_tropic_W', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [12]:
# meridional mean at the end in case it fails i dont loose anoms
mean_Uanom = Uanom.mean(dim='lat')
mean_Uanom.to_netcdf('/glade/work/acruz/E3SMv2LE/Uanom_tropic_slice.nc')

In [14]:
mean_Wanom = Wanom.mean(dim='lat') 
mean_Wanom.to_netcdf('/glade/work/acruz/E3SMv2LE/Wanom_tropic_slice.nc')

In [15]:
client.shutdown()

In [ ]:
quit()